# Lab 10: Path Planning on a Grid
## EE0849 Introduction to Robotics

In this lab you build the **plan** half of the sense-plan-act loop. You will:

1. Inflate obstacles by the robot's radius (C-space).
2. Implement Dijkstra on the 8-connected grid.
3. Extend it to A* with the octile heuristic.
4. Smooth the resulting grid path into sparse waypoints via line-of-sight shortcutting.

The notebook is self-contained --- a helper builds a test map so you do not need the Lab 9 output.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq
from scipy.ndimage import distance_transform_edt
from skimage.draw import line


---
## Part 1 --- C-space Inflation

A robot has physical size. We borrow a trick from configuration-space planning: grow every obstacle by the robot's radius, then pretend the robot is a single point. Formally this is the Minkowski sum `C_obs = Obstacle (+) disk(r)`.

On a grid we compute it with a **Euclidean distance transform**: for every free cell, find the distance (in cells) to the nearest occupied cell, then threshold at the robot radius.


### Step 1.1 --- Build the test map

In [ ]:
def make_test_map(M=100, N=100):
    """Binary occupancy grid: 0 = free, 1 = occupied.

    A 100x100 room with walls, an L-shaped obstacle, and one internal divider
    to force the planner around a corner.
    """
    grid = np.zeros((M, N), dtype=np.uint8)
    # Outer walls
    grid[0, :] = 1; grid[-1, :] = 1
    grid[:, 0] = 1; grid[:, -1] = 1
    # L-shaped obstacle in the middle
    grid[40:65, 40:65] = 1
    grid[50:65, 50:65] = 0
    # Internal divider with a gap
    grid[20:55, 75] = 1
    grid[75, 25:75] = 1
    return grid


grid = make_test_map()
M, N = grid.shape
print(f'Test map: {M} x {N} cells,  {grid.sum()} occupied')

plt.figure(figsize=(6, 6))
plt.imshow(grid.T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
plt.title('Raw occupancy grid')
plt.xlabel('i (x-index)'); plt.ylabel('j (y-index)')
plt.show()


### Step 1.2 --- Inflate the map

Use `scipy.ndimage.distance_transform_edt` on the mask of **free** cells. The output is the distance (in cells) from each free cell to the nearest occupied cell. Threshold at the robot radius.

Robot radius: 0.2 m. Grid resolution: 0.1 m. So `r_cells = 2`.

In [ ]:
# TODO:
#   1. Build a bool mask `free = (grid == 0)`.
#   2. Call distance_transform_edt(free) to get the distance from every free cell
#      to the nearest occupied cell (in cells, not metres).
#   3. Threshold: inflated = (dist <= r_cells).astype(np.uint8)
#   4. Return the inflated grid.

def inflate(grid, r_cells):
    free = ...                         # bool mask
    dist = ...                         # distance transform
    inflated = ...                     # threshold
    return inflated


r_cells = 2   # for a 0.2 m robot on a 0.1 m grid
inflated = inflate(grid, r_cells)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(grid.T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
axes[0].set_title('Raw map')
axes[1].imshow(inflated.T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
axes[1].set_title(f'Inflated (r = {r_cells} cells)')
for ax in axes:
    ax.set_xlabel('i'); ax.set_ylabel('j')
plt.tight_layout(); plt.show()

print(f'Free cells before inflation: {(grid == 0).sum()}')
print(f'Free cells after  inflation: {(inflated == 0).sum()}')


---
## Part 2 --- Dijkstra's Algorithm

Treat the inflated grid as a graph: each free cell is a node, each 8-connected move is an edge with cost 1 (cardinal) or sqrt(2) (diagonal). Dijkstra explores cells in order of increasing distance from the start, and stops as soon as the goal is popped.


### Step 2.1 --- 8-connected neighbors

In [ ]:
def neighbors_8(i, j, M, N):
    """Yield (ni, nj, step_cost) for the 8-connected neighbors of (i, j) that are in bounds.

    Cardinal moves cost 1; diagonal moves cost sqrt(2).
    """
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            ni, nj = i + di, j + dj
            if 0 <= ni < M and 0 <= nj < N:
                cost = np.sqrt(2) if (di != 0 and dj != 0) else 1.0
                yield ni, nj, cost


# Quick check
count = sum(1 for _ in neighbors_8(50, 50, 100, 100))
print(f'Interior cell has {count} neighbors  (expected 8)')
count = sum(1 for _ in neighbors_8(0, 0, 100, 100))
print(f'Corner cell has   {count} neighbors  (expected 3)')


### Step 2.2 --- Pick a start and goal

We will reuse these for Dijkstra and A*.

In [ ]:
start = (5, 5)
goal  = (95, 95)


### Step 2.3 --- Plot helper

In [ ]:
def plot_search(grid, path, closed_order, start, goal, title):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(grid.T, origin='lower', cmap='gray_r', vmin=0, vmax=1)

    # Expanded cells, colored by order
    visited = closed_order > 0
    if visited.any():
        ax.imshow(np.where(visited, closed_order, np.nan).T,
                  origin='lower', cmap='viridis', alpha=0.55,
                  vmin=1, vmax=closed_order.max())

    if path:
        pi = [p[0] for p in path]
        pj = [p[1] for p in path]
        ax.plot(pi, pj, color='red', lw=2, label='path')
    ax.scatter([start[0]], [start[1]], c='lime', s=80, edgecolors='k', zorder=5, label='start')
    ax.scatter([goal[0]],  [goal[1]],  c='orange', s=80, edgecolors='k', zorder=5, label='goal')
    ax.legend(loc='upper left')
    ax.set_title(title)
    plt.show()


### Step 2.4 --- Implement Dijkstra

In [ ]:
# TODO: implement Dijkstra on the 8-connected grid.
#
# Steps:
#   1. Initialise g as a (M, N) array filled with +inf. Set g[start] = 0.
#   2. Use a heapq priority queue. Push (0.0, start).
#   3. While the queue is non-empty: pop (d, u). If d > g[u], skip (stale entry).
#      Record the pop order in closed_order. If u == goal, break.
#   4. For each neighbor (ni, nj, step) from neighbors_8:
#        - Skip if grid[ni, nj] is occupied.
#        - Compute nd = d + step. If nd < g[ni, nj], relax and push.
#   5. Reconstruct the path using the parent dict.
#   6. Return (path, cost, g, closed_order).

def dijkstra(grid, start, goal):
    M_, N_ = grid.shape
    INF = float('inf')
    g = np.full((M_, N_), INF)
    parent = {}
    closed_order = np.zeros((M_, N_), dtype=np.int32)
    counter = 0

    # TODO: initialise and run the search loop here.

    path = []   # TODO: reconstruct
    return path, g[goal], g, closed_order


# Once implemented, uncomment:
# path, cost, g, order = dijkstra(inflated, start, goal)
# plot_search(inflated, path, order, start, goal, f'Dijkstra  --  cost = {cost:.2f}')


---
## Part 3 --- A* with the Octile Heuristic

Dijkstra has no idea where the goal is; it fans out equally in every direction. A* biases the search toward the goal by adding a heuristic `h(n)` to the priority. If `h(n)` never overestimates the true cost-to-go (admissible), A* finds the optimal path while expanding far fewer cells.

On an 8-connected grid the tightest admissible heuristic is the **octile distance**, which is the exact shortest path in empty space.


### Step 3.1 --- Octile heuristic

In [ ]:
# TODO: octile distance on an 8-connected grid with cardinal=1, diagonal=sqrt(2).
#
# Hint: let di = |a0 - b0|, dj = |a1 - b1|.
#       h = (di + dj) + (sqrt(2) - 2) * min(di, dj)

def octile(a, b):
    di = ...
    dj = ...
    return ...


# Sanity checks to write: octile((0,0), (5,5)) should be 5*sqrt(2).


### Step 3.2 --- A*

Copy your Dijkstra and change only the priority used when pushing onto the queue: `f = g + h` instead of just `g`.

In [ ]:
# TODO: adapt your Dijkstra to A*.
#
# Only two things change:
#   1. When you push an entry onto the queue, the priority is f = g + h,
#      where h = octile(current, goal). But you still store the actual g cost
#      in the entry so the relaxation check uses g, not f.
#   2. The initial push uses priority = octile(start, goal) and d = 0.
#
# Everything else (neighbors, relaxation, path reconstruction) is the same.

def astar(grid, start, goal):
    M_, N_ = grid.shape
    INF = float('inf')
    g = np.full((M_, N_), INF)
    parent = {}
    closed_order = np.zeros((M_, N_), dtype=np.int32)
    counter = 0

    # TODO: run A* here.

    path = []
    return path, g[goal], g, closed_order


### Step 3.3 --- Compare Dijkstra vs. A*

Both should return paths of identical cost. A* should expand far fewer cells --- visually, the 'fan' becomes a 'beam'.

In [ ]:
path_d, cost_d, _, order_d = dijkstra(inflated, start, goal)
path_a, cost_a, _, order_a = astar(inflated, start, goal)

n_d = int((order_d > 0).sum())
n_a = int((order_a > 0).sum())

print(f'Dijkstra: cost = {cost_d:.3f},  cells expanded = {n_d}')
print(f'A*      : cost = {cost_a:.3f},  cells expanded = {n_a}')
print(f'Speed-up: A* expanded {100 * n_a / n_d:.1f}% as many cells as Dijkstra')

plot_search(inflated, path_d, order_d, start, goal,
            f'Dijkstra  --  {n_d} cells expanded')
plot_search(inflated, path_a, order_a, start, goal,
            f'A* (octile)  --  {n_a} cells expanded')


---
## Part 4 --- Path Smoothing via Line-of-Sight

The raw grid path zigzags because the search is restricted to 8 discrete directions. A real robot can drive between any two points as long as the straight line is clear. We turn the dense path into a handful of waypoints by repeatedly skipping ahead to the farthest cell that still has line-of-sight from the current one.


### Step 4.1 --- Line of sight

In [ ]:
# TODO:
#   1. Use skimage.draw.line to get (ii, jj) arrays of cells on the Bresenham line
#      from a to b.
#   2. Return True iff grid[ii, jj] is 0 everywhere along that line.

def has_line_of_sight(grid, a, b):
    ii, jj = ...
    return ...


### Step 4.2 --- Greedy shortcut

In [ ]:
# TODO: greedy line-of-sight shortcut.
#
# Walk the input `path`. From each kept waypoint i, search BACKWARDS from the end
# for the farthest index j > i such that has_line_of_sight(path[i], path[j]) is True.
# Append path[j] to the output and set i = j.

def shortcut_path(grid, path):
    if not path:
        return path
    out = [path[0]]
    # TODO
    return out


### Step 4.3 --- Visualize

In [ ]:
smoothed = shortcut_path(inflated, path_a)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(inflated.T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
if path_a:
    pi = [p[0] for p in path_a]
    pj = [p[1] for p in path_a]
    ax.plot(pi, pj, color='red', lw=1.5, alpha=0.6, label=f'raw path ({len(path_a)} cells)')
if smoothed:
    si = [p[0] for p in smoothed]
    sj = [p[1] for p in smoothed]
    ax.plot(si, sj, color='tab:blue', lw=2, label=f'smoothed ({len(smoothed)} waypoints)')
    ax.scatter(si, sj, c='tab:blue', s=40, edgecolors='k', zorder=5)
ax.scatter([start[0]], [start[1]], c='lime', s=80, edgecolors='k', zorder=6, label='start')
ax.scatter([goal[0]],  [goal[1]],  c='orange', s=80, edgecolors='k', zorder=6, label='goal')
ax.legend(loc='upper left')
ax.set_title('Raw grid path vs. line-of-sight shortcut')
plt.show()

print(f'Raw path has {len(path_a)} cells')
print(f'Smoothed path has {len(smoothed)} waypoints')
print(f'Compression: {100 * len(smoothed) / max(len(path_a), 1):.1f}%')


---
## Wrap-up

You now have a pipeline:

1. Occupancy grid (Lab 9).
2. C-space inflation by the robot radius (this lab, Part 1).
3. A* search on the inflated grid (Parts 2-3).
4. Line-of-sight shortcutting to sparse waypoints (Part 4).

Those waypoints are the input to Week 9's trajectory controller --- the `act` half of the loop.
